In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import argparse
import datetime
import json
import os
import random

import numpy as np
import torch
import wandb
from accelerate.commands.config.update import description
from evals import compute_metrics, generate_eval_table
from pefts import get_model
from preprocess_data import preprocess_dataset
from transformers import (
    AutoModelForMaskedLM,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from utils import printd


def set_seed(seed=42) -> None:
    """Set all seeds to make results reproducible (deterministic mode).
    When seed is a false-y value or not supplied, disables deterministic mode."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


from dotenv import load_dotenv

load_dotenv()

True

In [3]:
config_path = "config.json"
data_src = "local"
gpu_index_to_use = "None"
train_techs = []
n_rows = 100

In [4]:

with open("config.json", "r") as file:
    config = json.load(file)
# config["termina_args"] = vars(args)

current_datetime = datetime.datetime.now()
formatted_datetime = current_datetime.strftime("%Y%m%d_%H-%M-%S")


model_output_dir = os.path.join(
    config.get("output").get("dir"),
    str(formatted_datetime),
)
os.makedirs(model_output_dir, exist_ok=True)
file = open(os.path.join(model_output_dir, config.get("output").get("log")), "w")

if gpu_index_to_use is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = gpu_index_to_use

printd("*" * 10 + "GPUs" + "*" * 10, file=file)
for i in range(torch.cuda.device_count()):
    printd(torch.cuda.get_device_properties(i).name, file=file)
printd("*" * 30, file=file)

# assert os.getenv("WANDB_LOG_MODEL") == "end"
# wandb.login(key=os.getenv("WANDB_API_KEY"))
# wandb.init(project="mlm-fine-tuning")


**********GPUs**********
******************************


In [5]:
printd("*" * 10 + "Started DataPreprocessing" + "*" * 10, file=file)
lm_dataset, tokenizer, data_collator = preprocess_dataset(
    config.get("input"),
    data_src,
    n_rows,
)
lm_dataset

**********Started DataPreprocessing**********


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 418
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 58
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 46
    })
})

In [6]:

model = get_model(config, train_techs, file)

training_args = TrainingArguments(
    output_dir=f"{config.get('output').get('model_backups_path')}timestamp_{formatted_datetime}/{config.get('input').get('model').get('hf')}/",
    **config.get("TrainingArguments"),
    label_names=["labels"],  # https://github.com/huggingface/peft/issues/1120
)
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=config.get("additional_training_config").get(
        "training_patience",
    ),
)
trainer_config = dict(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["validation"],
    data_collator=data_collator,
    # compute_metrics=compute_metrics,
    callbacks=[early_stopping],
    tokenizer=tokenizer,
)
trainer = Trainer(**trainer_config)
trainer.can_return_loss = True

# wandb.watch(model, log="all")
# update the config to wandb
custom_cfg = dict(
    **config,
    train_size=sum([len(i) for i in lm_dataset["train"]["input_ids"]]),
    val_size=sum([len(i) for i in lm_dataset["validation"]["input_ids"]]),
    test_size=sum([len(i) for i in lm_dataset["test"]["input_ids"]]),
)
custom_cfg["data_size"] = (
    custom_cfg["train_size"] + custom_cfg["val_size"] + custom_cfg["test_size"]
)
# wandb.config.update(custom_cfg)

/tmp/ipykernel_32247/3581520503.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(**trainer_config)


In [7]:
printd("*" * 10 + "Started Training" + "*" * 10, file=file)
trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


**********Started Training**********


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sajil (nasa-impact). Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


TrainOutput(global_step=27, training_loss=2.0238850911458335, metrics={'train_runtime': 46.2374, 'train_samples_per_second': 9.04, 'train_steps_per_second': 0.584, 'total_flos': 27511240914432.0, 'train_loss': 2.0238850911458335, 'epoch': 1.0})

In [8]:
# # tokenizer.mask_token, tokenizer.mask_token_id, tokenizer.encode(tokenizer.mask_token), tokenizer.decode(tokenizer.encode(tokenizer.mask_token))
# import random

# import random

# def mask_random_token(tokenized_example, tokenizer):
#     """
#     Randomly masks a token in the input_ids of a tokenized example and returns the masked token.
    
#     Args:
#         tokenized_example (dict): A dictionary containing tokenized data with "input_ids".
#         tokenizer (PreTrainedTokenizer): The tokenizer to convert tokens to ids and vice versa.
    
#     Returns:
#         dict: The updated example with one token masked and the original masked token.
#     """
#     input_ids = tokenized_example["input_ids"]
    
#     # Randomly select a position to mask (excluding special tokens like [CLS], [SEP], [PAD])
#     maskable_positions = [
#         i for i in range(len(input_ids)) 
#         if input_ids[i] not in [tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id]
#     ]
    
#     if maskable_positions:
#         mask_index = random.choice(maskable_positions)
#         original_token = input_ids[mask_index]  # Store the original token
#         input_ids[mask_index] = tokenizer.mask_token_id  # Replace with [MASK] token
    
#     # Store the original masked token in the example for future use (e.g., as target)
#     tokenized_example["input_ids"] = input_ids
#     tokenized_example["masked_token_str"] = tokenizer.decode([original_token])  # Decode the masked token back to string
#     return tokenized_example



# masked_dataset = lm_dataset["test"].map(
#     lambda example: mask_random_token(example, tokenizer)
# )


In [9]:
decoded_texts = [tokenizer.decode(example["input_ids"]) for example in masked_dataset]


NameError: name 'masked_dataset' is not defined

In [ ]:
decoded_texts[0]

"<s> toolkit  contests  webinars  get involved  resources  support  print essay  2020 essay contest finalist  entry by korina cortezano  how we process earth materials  all life on earth is dependent on earth's materials, it's what connects humans to the planet. but when we don't properly create balance in what we take out and put back in our earth, we<mask> slowly begin to lose what we depend on most.  earth's materials are grouped into four basic categories: minerals, rocks, soil, and water. these elements are naturally occurring, and support the survival of plants and animals,"

In [ ]:
print(masked_dataset[0])  # Check the first example
print(tokenizer.decode(masked_dataset[0]["input_ids"]))

{'input_ids': [0, 30749, 149, 261, 13070, 149, 257, 2692, 1897, 149, 4022, 1844, 149, 4444, 149, 1813, 149, 16330, 22082, 149, 5568, 22082, 15467, 45575, 149, 6068, 309, 354, 3692, 506, 575, 193, 68, 5687, 149, 665, 257, 949, 6134, 1374, 149, 489, 1687, 280, 6134, 271, 4157, 280, 6134, 666, 1374, 16, 442, 666, 2298, 24983, 4267, 228, 192, 18911, 18, 635, 809, 257, 2380, 2611, 11975, 5872, 5297, 199, 2298, 257, 3855, 704, 215, 3782, 1656, 199, 764, 6134, 16, 257, 50264, 11701, 4954, 228, 15509, 2298, 257, 3948, 280, 837, 18, 149, 6134, 666, 1374, 336, 10600, 801, 1338, 4659, 4765, 30, 17144, 16, 17312, 16, 3250, 16, 215, 1351, 18, 517, 3982, 336, 10088, 6636, 16, 215, 1813, 192, 1905, 206, 2939, 215, 2361, 16], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [10]:
model_save_loc = os.path.join(
    model_output_dir,
    config.get("output").get("final_model_path"),
)
os.makedirs(model_save_loc, exist_ok=True)
model.save_pretrained(model_save_loc)
tokenizer.save_pretrained(model_save_loc)

('./model_outputs/20241206_03-34-37/model/tokenizer_config.json',
 './model_outputs/20241206_03-34-37/model/special_tokens_map.json',
 './model_outputs/20241206_03-34-37/model/vocab.json',
 './model_outputs/20241206_03-34-37/model/merges.txt',
 './model_outputs/20241206_03-34-37/model/added_tokens.json',
 './model_outputs/20241206_03-34-37/model/tokenizer.json')

In [ ]:
lm_dataset["test"].num_rows

46

In [28]:
from utils import generate_inference
x = generate_inference(lm_dataset["test"], tokenizer, model_save_loc)
x

Device set to use cpu


,sequence,target,top1,top2,top3
0,toolkit contests webinars get involved re...,will,"{'score': 0.4985, 'token_str': ' will', 'token...","{'score': 0.1509, 'token_str': ''ll', 'token':...","{'score': 0.1487, 'token_str': ' can', 'token'..."
1,"the structure of the land, and the evolution ...",our,"{'score': 0.9956, 'token_str': ' our', 'token'...","{'score': 0.0008, 'token_str': ' their', 'toke...","{'score': 0.0008, 'token_str': ' us', 'token':..."
2,something that could potentially harm our com...,planet,"{'score': 0.2479, 'token_str': ' community', '...","{'score': 0.1613, 'token_str': ' environment',...","{'score': 0.0648, 'token_str': ' planet', 'tok..."
3,possible if we all leave positive footprints ...,,"{'score': 0.8192, 'token_str': ' ', 'token': 149}","{'score': 0.1445, 'token_str': ',', 'token': 16}","{'score': 0.0081, 'token_str': ' ', 'token': ..."
4,toolkit contests webinars get involved re...,with,"{'score': 0.3624, 'token_str': ' for', 'token'...","{'score': 0.2391, 'token_str': ' of', 'token':...","{'score': 0.1672, 'token_str': ' and', 'token'..."
5,"varies in thickness and distribution, and is ...",with,"{'score': 0.6057, 'token_str': ' with', 'token...","{'score': 0.1658, 'token_str': ' in', 'token':...","{'score': 0.118, 'token_str': ' covering', 'to..."
6,ked around the bottom* roasting pan or simila...,precipitation,"{'score': 0.417, 'token_str': ' precipitation'...","{'score': 0.1581, 'token_str': ' moisture', 't...","{'score': 0.0349, 'token_str': ' pressure', 't..."
7,"with half of the gravel, all the ice, and the...",,"{'score': 1.0, 'token_str': ' ', 'token': 149}","{'score': 0.0, 'token_str': ' ', 'token': 723}","{'score': 0.0, 'token_str': '</s>', 'token': 2}"
8,on communities? how does thawing permafrost ...,more,"{'score': 0.9916, 'token_str': ' more', 'token...","{'score': 0.0011, 'token_str': ' information',...","{'score': 0.001, 'token_str': ' much', 'token'..."
9,) youtube (opens in a new tab) instagram (op...,@,"{'score': 0.9675, 'token_str': '@', 'token': 36}","{'score': 0.0226, 'token_str': '.', 'token': 18}","{'score': 0.0049, 'token_str': ' @', 'token': ..."


In [22]:
from transformers import pipeline

# Load the fill-mask pipeline
mask_filler = pipeline("fill-mask", model=model_save_loc, tokenizer=model_save_loc)


results = mask_filler(decoded_texts, top_k=3)

print(results)

Device set to use cpu


NameError: name 'decoded_texts' is not defined

In [19]:
results

[[{'score': 0.4985329210758209,
   'token': 1124,
   'token_str': ' will',
   'sequence': " toolkit  contests  webinars  get involved  resources  support  print essay  2020 essay contest finalist  entry by korina cortezano  how we process earth materials  all life on earth is dependent on earth's materials, it's what connects humans to the planet. but when we don't properly create balance in what we take out and put back in our earth, we will slowly begin to lose what we depend on most.  earth's materials are grouped into four basic categories: minerals, rocks, soil, and water. these elements are naturally occurring, and support the survival of plants and animals,"},
  {'score': 0.15092980861663818,
   'token': 12168,
   'token_str': "'ll",
   'sequence': " toolkit  contests  webinars  get involved  resources  support  print essay  2020 essay contest finalist  entry by korina cortezano  how we process earth materials  all life on earth is dependent on earth's materials, it's what conne

In [ ]:

len(decoded_texts), len(results), 

(46, 46)

In [18]:
test_set = lm_dataset["test"]
predictions = trainer.predict(test_set)

In [19]:
lm_dataset["test"]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 15
})

In [25]:
import pandas as pd
from transformers import Trainer

def generate_inference_df_with_masking(trainer, lm_dataset, tokenizer, data_collator):
    # Apply the data collator to mask the test dataset
    test_dataset = lm_dataset["test"].map(
        lambda batch: data_collator(batch),
        batched=True,
    )

    # Get predictions on the masked test set
    predictions, labels, _ = trainer.predict(test_dataset)
    predictions = np.argmax(predictions, axis=-1)

    # Decode the input sequences
    input_texts = [
        tokenizer.decode(input_ids, skip_special_tokens=True) 
        for input_ids in test_dataset["input_ids"]
    ]

    # Process labels and predictions
    label_texts = []
    predicted_texts = []

    for input_ids, label_ids, pred_ids in zip(test_dataset["input_ids"], labels, predictions):
        masked_input = tokenizer.convert_ids_to_tokens(input_ids)
        
        # Filter labels and predictions
        filtered_labels = [
            label if label != -100 else tokenizer.pad_token_id for label in label_ids
        ]
        filtered_predictions = [
            pred if label != -100 else tokenizer.pad_token_id 
            for label, pred in zip(label_ids, pred_ids)
        ]

        # Decode into human-readable tokens
        label_texts.append(tokenizer.decode(filtered_labels, skip_special_tokens=True))
        predicted_texts.append(tokenizer.decode(filtered_predictions, skip_special_tokens=True))

    # Create DataFrame to store results
    df = pd.DataFrame({
        'input_with_masking': input_texts,
        'labels': label_texts,
        'predictions': predicted_texts,
    })

    return df



# After training, call the function to get the inference results
inference_df = generate_inference_df_with_masking(trainer, lm_dataset, tokenizer, data_collator)

# # Log the DataFrame to wandb
# wandb.log({"inference": wandb.Table(dataframe=inference_df)})

# # Optionally, save the results to a CSV
# inference_df.to_csv(os.path.join(model_output_dir, "test_inference.csv"), index=False)
inference_df

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

KeyError: 0

In [48]:
from torch.utils.data import DataLoader

test_dataloader = DataLoader(lm_dataset['test'], batch_size=16, collate_fn=data_collator)

for b in test_dataloader:
    # Print batch details
    print(b.keys())  # Uncomment to see available keys
    print(b["input_ids"].shape)
    print(b["labels"])
    print("-" * 14)

    

    # mask_token_id = tokenizer.mask_token_id  # Token ID for [MASK]

    # # Iterate over the batch
    # for input_ids, labels in zip(b["input_ids"], b["labels"]):
    #     # Replace -100 in the labels with [MASK]
    #     input_ids_with_mask = [
    #         mask_token_id if label == -100 else token_id
    #         for token_id, label in zip(input_ids.tolist(), labels.tolist())
    #     ]

    #     # Decode the modified input_ids
    #     original_text = tokenizer.decode(input_ids_with_mask, skip_special_tokens=True)
    #     print(original_text)

    break


dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([15, 512])
tensor([[-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],
        ...,
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100]])
--------------


In [28]:
tokenizer.decode(mask_token_id)

'<mask>'

In [14]:


# Inspect masked tokens and input
for i, (input_ids, labels) in enumerate(zip(masked_batch["input_ids"], masked_batch["labels"])):
    # Decode original input and masked output
    original_text = tokenizer.decode(input_ids, skip_special_tokens=True)
    masked_text = tokenizer.decode(
        [label if label != -100 else input_ids[idx] for idx, label in enumerate(labels)],
        skip_special_tokens=True,
    )

    print(f"Original Input {i + 1}: {original_text}")
    print(f"Masked Input {i + 1}: {masked_text}")
    print("-" * 50)

NameError: name 'masked_batch' is not defined